# CubeSandbox 远程演示（Windows → 鲲鹏）

本机 Jupyter 只是遥控器；沙箱在鲲鹏 MicroVM 中执行。

使用前：复制 `.env.example` 为 `.env`，填写 `CUBE_TEMPLATE_ID`；从鲲鹏拷贝 `rootCA.pem`（见 `README_zh.md`）。

## Cell 1 — 加载配置

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from e2b_code_interpreter import Sandbox

load_dotenv(Path(".env"))

for k in ["E2B_API_URL", "E2B_API_KEY", "CUBE_TEMPLATE_ID", "SSL_CERT_FILE", "REQUESTS_CA_BUNDLE"]:
    print(f"{k} = {os.environ.get(k)}")

## Cell 2 — 创建沙箱

In [ ]:
sbx = Sandbox.create(template=os.environ["CUBE_TEMPLATE_ID"], timeout=600)
print("sandbox_id =", sbx.sandbox_id)
print("创建成功：隔离 MicroVM 已在鲲鹏上拉起")

## Cell 3 — 架构 / Python 版本

In [ ]:
r = sbx.run_code("import platform, sys\nprint(platform.machine())\nprint(sys.version)")
print(r)

## Cell 4 — 探测数据科学包（True=已装）

In [ ]:
r = sbx.run_code(
    "import importlib.util as u\n"
    "for m in ['numpy','pandas','matplotlib','sklearn']:\n"
    "    print(m, bool(u.find_spec(m)))\n"
)
print(r)

## Cell 5 — 简单计算（证明在沙箱内执行）

In [ ]:
r = sbx.run_code(
    "s = sum(range(1, 101))\n"
    "print('1+...+100 =', s)\n"
    "s"
)
print(r)

## Cell 6 — Shell（可选）

In [ ]:
print(sbx.commands.run("uname -a").stdout)
print(sbx.commands.run("df -h / | tail -1").stdout)

## Cell 7 — 销毁沙箱（演示结束必跑）

In [ ]:
sbx.kill()
print("sandbox destroyed")